# Data initialization (model-ready)

This notebook loads `processed_data/f2_combined_raw.csv` and derives consistent, model-ready columns using the proxy guidance in `raw_data/README.md`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW = ROOT / "raw_data"
PROCESSED = ROOT / "processed_data"

combined_path = PROCESSED / "f2_combined_raw.csv"
combined_path

PosixPath('/Users/binayakjha/Desktop/clausura/processed_data/f2_combined_raw.csv')

In [2]:
df = pd.read_csv(combined_path, low_memory=False)
df.shape, df.columns.tolist()[:15]

((20184, 16),
 ['UNITID',
  'year',
  'F2D01',
  'F2D16',
  'F2A02',
  'F2A03',
  'F2B02',
  'F2B04',
  'F2A04',
  'F2A05B',
  'F2I03',
  'F2I05',
  'F2I07',
  'F2H01',
  'F2H02'])

In [3]:
df.head(3)

,UNITID,year,F2D01,F2D16,F2A02,F2A03,F2B02,F2B04,F2A04,F2A05B,F2I03,F2I05,F2I07,F2H01,F2H02,F2FHA
0,100690,2014,6765224,10236876,13848241.0,3019785.0,8348077.0,1888799.0,10242446.0,411205.0,NaN,NaN,NaN,174805.0,174804.0,1
1,100937,2014,13489284,48876673,195685360.0,72417000.0,45134655.0,3742018.0,47128931.0,15752885.0,NaN,NaN,NaN,50042970.0,53921439.0,1
2,101073,2014,2843422,12860451,17760303.0,3817334.0,13032162.0,-171711.0,11462292.0,815096.0,NaN,NaN,NaN,NaN,NaN,2


In [4]:
def to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

NUMERIC_COLS = [
    "F2D01", "F2D16",
    "F2A02", "F2A03",
    "F2B02", "F2B04",
    "F2A04", "F2A05B",
    "F2I03", "F2I05", "F2I07",
    "F2H01", "F2H02",
]
for c in NUMERIC_COLS:
    if c in df.columns:
        df[c] = to_numeric(df[c])

# Ensure types
df["UNITID"] = pd.to_numeric(df["UNITID"], errors="coerce").astype("Int64")
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

df[["UNITID", "year"]].isna().sum().to_dict()

{'UNITID': 0, 'year': 0}

In [ ]:
# Model-ready proxies and derived fields
# raw_data/README.md guidance:
# - Total expenses: use F2I07 when present, else F2B02
# - Change in net assets: use F2I03 when present, else F2B04
# - Expendable net assets: use F2I05 when present, else (F2A04 + F2A05B)

df["total_expenses"] = df["F2I07"].where(df["F2I07"].notna(), df["F2B02"])
df["change_in_net_assets"] = df["F2I03"].where(df["F2I03"].notna(), df["F2B04"])
df["expendable_net_assets"] = df["F2I05"].where(df["F2I05"].notna(), df["F2A04"] + df["F2A05B"])

# Convenience ratios commonly used for financial health modeling
df["assets_to_liabilities"] = df["F2A02"] / df["F2A03"].replace({0: np.nan})
df["operating_margin"] = df["change_in_net_assets"] / df["F2D16"].replace({0: np.nan})
df["expenses_to_revenue"] = df["total_expenses"] / df["F2D16"].replace({0: np.nan})

# Endowment fields (if present)
df["endowment_boy"] = df.get("F2H01")
df["endowment_eoy"] = df.get("F2H02")

df[[
    "total_expenses", "change_in_net_assets", "expendable_net_assets",
    "assets_to_liabilities", "operating_margin", "expenses_to_revenue"
]].describe().T

,count,mean,std,min,25%,50%,75%,max
total_expenses,17889.0,1.436646e+08,6.477590e+08,1.100000e+04,7.004375e+06,2.819540e+07,7.245082e+07,1.513128e+10
change_in_net_assets,17889.0,2.453670e+07,2.766105e+08,-3.215838e+09,-3.196900e+05,7.841190e+05,6.708119e+06,1.320536e+10
expendable_net_assets,17889.0,2.891794e+08,1.869517e+09,-1.375834e+08,4.308180e+06,2.746803e+07,9.849586e+07,4.697878e+10
assets_to_liabilities,17838.0,9.438612e+03,8.268509e+05,1.413182e-02,2.434144e+00,3.746303e+00,6.526615e+00,8.381110e+07
operating_margin,17883.0,4.825276e-02,8.798311e-01,-4.835606e+01,-2.574826e-02,5.628207e-02,1.637486e-01,3.068841e+01
expenses_to_revenue,17883.0,9.921445e-01,1.164310e+00,-2.820326e+01,8.478313e-01,9.512016e-01,1.038852e+00,7.075699e+01


In [6]:
# Build a model-ready table
FEATURE_COLS = [
    "UNITID", "year",
    "F2D01", "F2D16", "F2A02", "F2A03",
    "total_expenses", "change_in_net_assets", "expendable_net_assets",
    "assets_to_liabilities", "operating_margin", "expenses_to_revenue",
    "endowment_boy", "endowment_eoy",
]

model_df = df[FEATURE_COLS].copy()
model_df = model_df.dropna(subset=["UNITID", "year"]).sort_values(["UNITID", "year"], kind="mergesort")
model_df.shape, model_df.head(3)

((20184, 14),
       UNITID  year    F2D01     F2D16       F2A02      F2A03  total_expenses  \
 0     100690  2014  6765224  10236876  13848241.0  3019785.0       8348077.0   
 1883  100690  2015  6709019   8550133  12008292.0  3194663.0       8064960.0   
 3805  100690  2016  6819566   7854250  12111063.0  3157036.0       7713852.0   
 
       change_in_net_assets  expendable_net_assets  assets_to_liabilities  \
 0                1888799.0             10653651.0               4.585837   
 1883            -2014827.0              8638824.0               3.758860   
 3805              140398.0              8779222.0               3.836213   
 
       operating_margin  expenses_to_revenue  endowment_boy  endowment_eoy  
 0             0.184509             0.815491       174805.0       174804.0  
 1883         -0.235649             0.943256       174804.0       174805.0  
 3805          0.017875             0.982125       174805.0       174804.0  )

In [7]:
out_path = PROCESSED / "f2_model_ready.csv"
model_df.to_csv(out_path, index=False)
out_path

PosixPath('/Users/binayakjha/Desktop/clausura/processed_data/f2_model_ready.csv')